# Alba Residual Architecture: Paper-Grade Scaling Study (125M – 300M+ Params)

Notebook huấn luyện **từ đầu (From-Scratch Pretraining)** trên **`FineWeb-Edu` (`sample-10BT`)** phục vụ chuẩn bị **Paper (arXiv / Workshop / ICLR-NeurIPS)**:
- **`pre_norm`**: Pre-Norm chuẩn (`x + u`)
- **`pre_norm_isotime`**: Pre-Norm sâu hơn (`+25% layers`) để đối chứng công bằng theo wall-clock / compute
- **`mhc`**: Manifold-Constrained Hyper-Connections (`n=4` streams, Sinkhorn-Knopp `arXiv:2512.24880`)
- **`gor_v2`**: Zero-Param Geometric Orthogonal Residual (`phi <= 2`)
- **`attn_res_block`**: Block AttnRes (`N=8` / `N=9` / `N=10`)
- **`attn_res_gor_v2`**: **Block AttnRes + GOR-v2 (Đề xuất chính)**

### 4-Phase Paper Scaling Switchboard

| Phase Key | Purpose | Preset | Variants | Seeds | Optimizer | Decoupled Connector Args | Approx A100 Runtime |
|---|---|---|---|---|---|---|---|
| `phase1_geomuon_v21_gate` | Fast 40M 3-seed validation gate for GEO-Muon v2.1 with connector lr/wd decoupling | `ablation_40m` | `pre_norm,attn_res_gor_v2` | `42,137,2026` | `geomuon` | `--geomuon-connector-lr-mult 0.3 --geomuon-connector-wd 0.02` | ~25 mins |
| **`phase2_scale_125m_adamw`** *(Default)* | Primary paper scaling benchmark across 4 architectures under AdamW | `scale_125m` | `pre_norm,pre_norm_isotime,attn_res_block,attn_res_gor_v2` | `42` | `adamw` | *(None)* | ~4.5 hours |
| `phase3_scale_220m_adamw` | Deep 220M (18L) architectural scaling benchmark under AdamW | `scale_220m` | `pre_norm,attn_res_block,attn_res_gor_v2` | `42` | `adamw` | *(None)* | ~6.5 hours |
| `phase4_scale_125m_geomuon_v21` | Scale-125M benchmark under GEO-Muon v2.1 with decoupled connector | `scale_125m` | `pre_norm,attn_res_gor_v2` | `42` | `geomuon` | `--geomuon-connector-lr-mult 0.3 --geomuon-connector-wd 0.02` | ~2.5 hours |

**Các preset quy mô (`SCALE_PRESETS`):**
- `ablation_40m`: ~40M params (`d=384, L=12, T=512`), `60M` tokens (ablation nhanh + 3 seeds)
- `scale_125m`: **~124M params** (`d=768, L=12, T=1024`), `300M` tokens
- `scale_220m`: **~221M params** (`d=896, L=18, T=1024`, gradient checkpointing), `400M` tokens
- `scale_300m`: **~308M params** (`d=1024, L=20, T=1024`, gradient checkpointing), `500M` tokens

Hệ thống tự động **streaming token vào `uint16` memmap**, **lưu từng `(variant, seed)` vào Google Drive (nếu mount)** để Colab ngắt kết nối vẫn **resume** tiếp không mất công sức, đồng thời xuất sẵn **`paper_tables.tex`** (LaTeX), **`paper_figures.pdf/png`**, và đo **Zero-Shot Accuracy (`HellaSwag`, `ARC-Easy`, `LAMBADA`)**.


In [ ]:
import os
import torch

os.system("nvidia-smi")
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

os.system("pip install -q datasets transformers matplotlib numpy")

In [ ]:
import os

USE_GDRIVE = False
WORK_ROOT = "/content/alba_paper_work"
if os.path.isdir("/content/drive/MyDrive"):
    WORK_ROOT = "/content/drive/MyDrive/Alba_Paper_Scaling"
elif USE_GDRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK_ROOT = "/content/drive/MyDrive/Alba_Paper_Scaling"
    except Exception as err:
        print("Drive mount skipped (using local /content):", err)

os.makedirs(WORK_ROOT, exist_ok=True)
CACHE_DIR = f"{WORK_ROOT}/fineweb_cache"
OUTPUT_DIR = f"{WORK_ROOT}/results"
print("WORK_ROOT:", WORK_ROOT)

In [ ]:
import os
import subprocess

def run_live(cmd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}")

repo_dir = "/content/Alba-model"
if os.path.exists(repo_dir):
    subprocess.run(["rm", "-rf", repo_dir], check=True)
run_live(["git", "clone", "-b", "Khang", "https://github.com/potatips/Alba-model.git", repo_dir])
os.chdir(repo_dir)
run_live(["python3", "-u", "-m", "unittest", "gor_research.test_gor_research", "-v"])

### 4-Phase Switchboard Run Controller

Select the experimental phase by setting `PHASE` in the code cell below. The switchboard automatically configures the preset, architectures, seeds, optimizer, and decoupled connector hyperparameters.

| Phase Key | Preset | Variants | Seeds | Optimizer | Extra Args | Approx A100 Runtime |
|---|---|---|---|---|---|---|
| `phase1_geomuon_v21_gate` | `ablation_40m` | `pre_norm,attn_res_gor_v2` | `42,137,2026` | `geomuon` | `--geomuon-connector-lr-mult 0.3 --geomuon-connector-wd 0.02` | ~25 mins |
| **`phase2_scale_125m_adamw`** *(Default)* | `scale_125m` | `pre_norm,pre_norm_isotime,attn_res_block,attn_res_gor_v2` | `42` | `adamw` | *(None)* | ~4.5 hours |
| `phase3_scale_220m_adamw` | `scale_220m` | `pre_norm,attn_res_block,attn_res_gor_v2` | `42` | `adamw` | *(None)* | ~6.5 hours |
| `phase4_scale_125m_geomuon_v21` | `scale_125m` | `pre_norm,attn_res_gor_v2` | `42` | `geomuon` | `--geomuon-connector-lr-mult 0.3 --geomuon-connector-wd 0.02` | ~2.5 hours |

To switch phases, set `PHASE` to any key above. Set `CUSTOM_*` variables only if overriding default phase parameters.


In [ ]:
import os
import subprocess

if os.path.isdir("/content/Alba-model"):
    os.chdir("/content/Alba-model")

if "WORK_ROOT" not in globals():
    WORK_ROOT = "/content/alba_paper_work"
    if os.path.isdir("/content/drive/MyDrive"):
        WORK_ROOT = "/content/drive/MyDrive/Alba_Paper_Scaling"
    try:
        os.makedirs(WORK_ROOT, exist_ok=True)
    except OSError:
        pass

if "CACHE_DIR" not in globals():
    CACHE_DIR = f"{WORK_ROOT}/fineweb_cache"

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = f"{WORK_ROOT}/results"

PHASE_CONFIGS = {
    "phase1_geomuon_v21_gate": {
        "preset": "ablation_40m",
        "variants": "pre_norm,attn_res_gor_v2",
        "seeds": "42,137,2026",
        "optimizer": "geomuon",
        "extra_args": ["--geomuon-connector-lr-mult", "0.3", "--geomuon-connector-wd", "0.02"],
        "tag": "phase1_40m_geomuon_v21_gate",
    },
    "phase2_scale_125m_adamw": {
        "preset": "scale_125m",
        "variants": "pre_norm,pre_norm_isotime,attn_res_block,attn_res_gor_v2",
        "seeds": "42",
        "optimizer": "adamw",
        "extra_args": [],
        "tag": "phase2_125m_adamw",
    },
    "phase3_scale_220m_adamw": {
        "preset": "scale_220m",
        "variants": "pre_norm,attn_res_block,attn_res_gor_v2",
        "seeds": "42",
        "optimizer": "adamw",
        "extra_args": [],
        "tag": "phase3_220m_adamw",
    },
    "phase4_scale_125m_geomuon_v21": {
        "preset": "scale_125m",
        "variants": "pre_norm,attn_res_gor_v2",
        "seeds": "42",
        "optimizer": "geomuon",
        "extra_args": ["--geomuon-connector-lr-mult", "0.3", "--geomuon-connector-wd", "0.02"],
        "tag": "phase4_125m_geomuon_v21",
    },
}

PHASE = "phase2_scale_125m_adamw"

CUSTOM_PRESET = ""
CUSTOM_VARIANTS = ""
CUSTOM_SEEDS = ""
CUSTOM_OPTIMIZER = ""
CUSTOM_EXTRA_ARGS = None
CUSTOM_RUN_TAG = ""

if PHASE not in PHASE_CONFIGS:
    raise ValueError(f"Unknown PHASE: {PHASE}. Must be one of {list(PHASE_CONFIGS.keys())}")

phase_cfg = PHASE_CONFIGS[PHASE]
preset = CUSTOM_PRESET if CUSTOM_PRESET else phase_cfg["preset"]
variants = CUSTOM_VARIANTS if CUSTOM_VARIANTS else phase_cfg["variants"]
seeds = CUSTOM_SEEDS if CUSTOM_SEEDS else phase_cfg["seeds"]
optimizer = CUSTOM_OPTIMIZER if CUSTOM_OPTIMIZER else phase_cfg["optimizer"]
extra_args = list(CUSTOM_EXTRA_ARGS) if CUSTOM_EXTRA_ARGS is not None else list(phase_cfg["extra_args"])
run_tag = CUSTOM_RUN_TAG if CUSTOM_RUN_TAG else phase_cfg["tag"]
RUN_TAG = run_tag

STEPS = 0
CORPUS_TOKENS = 0
USE_COMPILE = True
SAVE_CKPT = False
CKPT_EVERY = 250
CKPT_MINS = 15.0
MAX_SESSION_MINS = 0.0

cmd = [
    "python3", "-u", "-m", "gor_research.paper_scaling_study",
    "--preset", preset,
    "--variants", variants,
    "--seeds", seeds,
    "--optimizer", optimizer,
    "--device", "cuda",
    "--cache-dir", CACHE_DIR,
    "--output-dir", f"{OUTPUT_DIR}/{RUN_TAG}",
    "--ckpt-every", str(CKPT_EVERY),
    "--ckpt-mins", str(CKPT_MINS),
]

if extra_args:
    cmd.extend(extra_args)
if MAX_SESSION_MINS > 0:
    cmd.extend(["--max-session-mins", str(MAX_SESSION_MINS)])
if STEPS > 0:
    cmd.extend(["--steps", str(STEPS)])
if CORPUS_TOKENS > 0:
    cmd.extend(["--corpus-tokens", str(CORPUS_TOKENS)])
if USE_COMPILE:
    cmd.append("--compile")
if SAVE_CKPT:
    cmd.append("--save-checkpoints")

print(f"Selected Phase: {PHASE}", flush=True)
print(f"Run Tag: {RUN_TAG}", flush=True)
print(f"Command: {' '.join(cmd)}", flush=True)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
code = proc.wait()
if code != 0:
    raise RuntimeError(f"Training failed with exit code {code}")


In [ ]:
import glob
import json
import os
from IPython.display import Image, display

run_dir = f"{OUTPUT_DIR}/{RUN_TAG}"
res_json = f"{run_dir}/paper_results.json"
if not os.path.exists(res_json):
    print("=== SESSION PAUSED (IN PROGRESS) ===")
    print("Completed runs:", sorted(glob.glob(f"{run_dir}/runs/*.json")))
    print("Saved mid-run checkpoints:", sorted(glob.glob(f"{run_dir}/step_checkpoints/*.pt")))
    print("Re-run training cell to continue from the exact saved step.")
else:
    with open(res_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("=== AGGREGATED SCALING SUMMARY ===")
    for v, r in data["summary"].items():
        ds = " | ".join(f"{k}={dv['mean']:.1f}%" for k, dv in r["downstream"].items())
        print(
            f"{v:18s} | {r['total_params']/1e6:5.1f}M | L={r['actual_layers']:2d} | "
            f"ValBPB={r['val_bpb']['mean']:.4f}±{r['val_bpb']['std']:.4f} | "
            f"OOD={r['ood_avg_bpb']['mean']:.4f} | Conflict={r['conflict_pct']['mean']:.2f}% | "
            f"{r['tok_per_sec']['mean']:,.0f} tok/s | {ds}"
        )

    print("\n=== LATEX TABLE ===")
    with open(f"{run_dir}/paper_tables.tex", "r", encoding="utf-8") as f:
        print(f.read())

    display(Image(filename=f"{run_dir}/paper_figures.png"))

In [ ]:
from google.colab import files

run_dir = f"{OUTPUT_DIR}/{RUN_TAG}"
for fname in ["paper_results.json", "paper_figures.png", "paper_figures.pdf", "paper_tables.tex", "paper_report.html"]:
    files.download(f"{run_dir}/{fname}")